# Smoke test: `opentelemetry_instrumentation_openai_agents_v2`

Exercises the public API of the OpenTelemetry instrumentation for the OpenAI Agents SDK: the `OpenAIAgentsInstrumentor` class, its declared dependencies, package metadata, and the instrument/uninstrument lifecycle.

**Requires the openai-agents package:** `pip install openai-agents` (or `pip install "opentelemetry-instrumentation-openai-agents-v2[instruments]"`). The instrumentor imports `agents.tracing` at import time, so without it every cell stops with `ModuleNotFoundError: No module named 'agents'`. Do not `pip install agents`; that is an unrelated project.

In [ ]:
# Example 1: Instantiate the OpenAIAgentsInstrumentor.
from opentelemetry.instrumentation.openai_agents import OpenAIAgentsInstrumentor

instrumentor = OpenAIAgentsInstrumentor()
print(f"Instrumentor type: {type(instrumentor).__name__}")
print(f"MRO: {[c.__name__ for c in type(instrumentor).__mro__]}")
assert instrumentor is not None

In [ ]:
# Example 2: Inspect instrumentation_dependencies (BaseInstrumentor API).
from opentelemetry.instrumentation.openai_agents import OpenAIAgentsInstrumentor

instrumentor = OpenAIAgentsInstrumentor()
deps = instrumentor.instrumentation_dependencies()
deps_list = list(deps)
print(f"instrumentation_dependencies() -> {deps_list}")
assert isinstance(deps_list, list)
# Should reference openai-agents in some form
assert any('openai' in str(d).lower() or 'agents' in str(d).lower() for d in deps_list) or len(deps_list) == 0

In [ ]:
# Example 3: Subclass relationship to BaseInstrumentor.
from opentelemetry.instrumentation.instrumentor import BaseInstrumentor
from opentelemetry.instrumentation.openai_agents import OpenAIAgentsInstrumentor

assert issubclass(OpenAIAgentsInstrumentor, BaseInstrumentor)
print(f"{OpenAIAgentsInstrumentor.__name__} is a subclass of BaseInstrumentor: True")

# Verify required abstract method overrides are present.
for method_name in ('_instrument', '_uninstrument', 'instrumentation_dependencies'):
    assert hasattr(OpenAIAgentsInstrumentor, method_name), f"missing {method_name}"
    print(f"  has method: {method_name}")

In [ ]:
# Example 4: Module-level package metadata.
import importlib.metadata

import opentelemetry.instrumentation.openai_agents as pkg

public_names = [n for n in dir(pkg) if not n.startswith('_')]
print(f"Public names in package: {public_names}")
assert 'OpenAIAgentsInstrumentor' in public_names

# The dist carries the -v2 name; its version is the authoritative one.
dist_version = importlib.metadata.version('opentelemetry-instrumentation-openai-agents-v2')
print(f"dist version: {dist_version}")
assert dist_version

In [ ]:
# Example 5: instrument()/uninstrument() roundtrip should not raise.
# With openai-agents installed, the instrumentor's lifecycle entry points must
# complete a full round trip without raising.
from opentelemetry.instrumentation.openai_agents import OpenAIAgentsInstrumentor

instrumentor = OpenAIAgentsInstrumentor()
if instrumentor.is_instrumented_by_opentelemetry:
    instrumentor.uninstrument()

instrumentor.instrument()
assert instrumentor.is_instrumented_by_opentelemetry
instrumentor.uninstrument()
assert not instrumentor.is_instrumented_by_opentelemetry
print("instrument()/uninstrument() roundtrip succeeded")

assert isinstance(instrumentor, OpenAIAgentsInstrumentor)
print("OK")